# Automated Interpretability

**Prerequisites**: [Notebook 03 — Sparse Autoencoders](03_sparse_autoencoders.ipynb), [Notebook 06 — Probing](06_probing.ipynb)

This notebook covers the emerging field of **automated interpretability** — using language models to automatically describe, validate, and understand features discovered by mechanistic interpretability tools.

Instead of manually inspecting thousands of features, we enlist LLMs themselves to interpret other LLMs, enabling interpretability to scale to models with millions of features.

## 1. The Auto-Interp Pipeline

Manual interpretability doesn't scale. If an SAE extracts 100,000 features, a human can't inspect them all. **Automated interpretability** uses LLMs to:

1. **Describe features**: Given a feature's top activating examples, ask an LLM "what concept does this feature detect?"
2. **Score descriptions**: Test whether the LLM's description predicts the feature's activations on new examples
3. **Discover circuits**: Use LLM-generated descriptions to identify functional relationships between features

The pipeline (introduced by Bills et al. 2023, OpenAI):

> **Note:** Bills et al. (2023) originally applied this pipeline to individual MLP neurons in GPT-2. The same approach was later adapted for SAE features by subsequent work (e.g., Anthropic's *Scaling Monosemanticity*).

1. Collect top-k activating dataset examples for a neuron/feature
2. Show these to an LLM: "Here are text excerpts where this feature activates strongly. What is this feature detecting?"
3. The LLM produces a natural language description
4. Validation: show the LLM NEW examples and ask it to predict whether the feature would activate
5. Score: correlation between predicted and actual activations

## 2. Implementing Auto-Interp from Scratch

We'll implement a simplified version using GPT-2 features and a simple scoring method (since we can't call an external LLM API from this notebook, we'll simulate the pipeline and show the data structures).

In [ ]:
import torch
import numpy as np
import json
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()

# We'll use the SAE from notebook 03 if available, otherwise work with raw neurons
# For this demo, we'll look at MLP neurons (no SAE needed)
hook_point = "blocks.6.mlp.hook_post"

# Corpus of diverse text
corpus = [
    "The president of the United States gave a speech at the White House today.",
    "Python is a programming language used for machine learning and data science.",
    "The stock market rose sharply after the Federal Reserve announced lower interest rates.",
    "Paris, the capital of France, is famous for the Eiffel Tower and fine cuisine.",
    "Quantum computing uses qubits instead of classical bits to perform calculations.",
    "The Great Barrier Reef is the world's largest coral reef system off Australia.",
    "Shakespeare wrote Romeo and Juliet, Hamlet, and many other famous plays.",
    "Machine learning models are trained using gradient descent on large datasets.",
    "The Amazon rainforest produces approximately twenty percent of the world's oxygen.",
    "Einstein's theory of general relativity describes gravity as curved spacetime.",
    "Basketball was invented by James Naismith in Springfield, Massachusetts in 1891.",
    "The human genome contains approximately three billion base pairs of DNA.",
    "Mozart composed his first symphony at the age of eight years old.",
    "Climate change is causing glaciers to melt and sea levels to rise globally.",
    "The internet was originally developed as ARPANET by the US Department of Defense.",
    "Photosynthesis converts carbon dioxide and water into glucose using sunlight energy.",
    "The Great Wall of China stretches over thirteen thousand miles across northern China.",
    "Neural networks are inspired by the structure of biological neurons in the brain.",
    "The Olympic Games originated in ancient Greece and were held every four years.",
    "Antibiotics like penicillin revolutionized medicine by treating bacterial infections effectively.",
]

# Collect activations for each text
all_acts = []
all_tokens = []
for text in corpus:
    _, cache = model.run_with_cache(text, names_filter=hook_point)
    acts = cache[hook_point][0].detach().cpu()  # (seq_len, d_mlp) — hook_post gives intermediate MLP neuron activations
    tokens = model.to_str_tokens(text)
    all_acts.append(acts)
    all_tokens.append(tokens)

print(f"Collected activations from {len(corpus)} texts")
print(f"MLP dimension: {all_acts[0].shape[-1]}")

In [ ]:
def get_top_activating_examples(neuron_idx, all_acts, all_tokens, top_k=10):
    """Find the top-k tokens that most strongly activate a given neuron."""
    activations = []
    for text_idx, (acts, tokens) in enumerate(zip(all_acts, all_tokens)):
        for pos in range(len(tokens)):
            activations.append({
                "text_idx": text_idx,
                "pos": pos,
                "token": tokens[pos],
                "activation": acts[pos, neuron_idx].item(),
                "context": "".join(tokens[max(0, pos-5):pos+5]),
            })
    
    # Sort by activation value
    activations.sort(key=lambda x: x["activation"], reverse=True)
    return activations[:top_k]

# Examine a few neurons
interesting_neurons = []
n_neurons = all_acts[0].shape[-1]

# Find neurons with interesting activation patterns (high variance, not always-on)
neuron_stats = []
for n in range(min(n_neurons, 500)):  # Check first 500 for speed
    all_values = torch.cat([a[:, n] for a in all_acts])
    if all_values.std() > 1.0 and all_values.mean() < 5.0:
        neuron_stats.append((n, all_values.std().item(), all_values.mean().item()))

neuron_stats.sort(key=lambda x: x[1], reverse=True)
print(f"Found {len(neuron_stats)} neurons with interesting activation patterns\n")

# Show top activating examples for top 3 interesting neurons
for neuron_idx, std, mean in neuron_stats[:3]:
    top_examples = get_top_activating_examples(neuron_idx, all_acts, all_tokens, top_k=8)
    print(f"=== Neuron {neuron_idx} (std={std:.2f}, mean={mean:.2f}) ===")
    for ex in top_examples:
        print(f"  [{ex['activation']:+.2f}] '{ex['token']}' in: ...{ex['context']}...")
    print()

## 3. Generating Feature Descriptions

In practice, you'd send the top activating examples to an LLM (GPT-4, Claude, etc.) with a prompt like:

```
Here are text excerpts where a neural network feature activates strongly.
The activating token is marked with [brackets].

1. "...the [president] of the United States..."  (activation: 8.3)
2. "...Prime [Minister] Johnson announced..."    (activation: 7.9)
3. "...elected [governor] of California..."       (activation: 7.1)
...

What concept or pattern does this feature detect?
Provide a concise description (1-2 sentences).
```

The LLM might respond: *"This feature detects political leadership titles and roles (president, minister, governor, etc.)"*

In [ ]:
# TODO: include_negatives parameter is accepted but not yet used.
# A future improvement would show low-activation examples for contrast.
def format_autointerp_prompt(neuron_idx, top_examples, include_negatives=True):
    """Format top activating examples into a prompt for an LLM."""
    
    prompt = "I'm analyzing a neural network feature (neuron). Below are text excerpts where this feature activates most strongly. The token with peak activation is shown in [brackets].\n\n"
    prompt += "TOP ACTIVATING EXAMPLES:\n"
    
    for i, ex in enumerate(top_examples[:8], 1):
        context = ex["context"]
        token = ex["token"]
        # Mark the activating token
        # NOTE: str.replace is fragile — it marks the first occurrence, which may
        # not be the activating token if the same substring appears earlier in context.
        # A more robust approach would use positional indexing.
        marked_context = context.replace(token, f"[{token}]", 1)
        prompt += f"{i}. \"{marked_context}\" (activation: {ex['activation']:.1f})\n"
    
    prompt += "\nBased on these examples, what concept or pattern does this feature detect?\n"
    prompt += "Provide a concise description (1-2 sentences)."
    
    return prompt

# Generate prompts for our interesting neurons
for neuron_idx, std, mean in neuron_stats[:3]:
    top_examples = get_top_activating_examples(neuron_idx, all_acts, all_tokens, top_k=8)
    prompt = format_autointerp_prompt(neuron_idx, top_examples)
    print(f"=== Auto-Interp Prompt for Neuron {neuron_idx} ===")
    print(prompt)
    print("\n" + "="*60 + "\n")

## 4. Scoring Descriptions (Validation)

A description is only useful if it **predicts** the feature's behavior. The scoring protocol:

1. Take the LLM-generated description
2. Show the LLM NEW examples (not used for generating the description)
3. For each example, ask: "Given that this feature detects [description], would it activate on this token? Rate 0-10."
4. Compare LLM predictions with actual activations
5. Score = correlation(predicted, actual)

**Scoring metrics:**
- **Detection score**: Can the description predict which tokens activate the feature? (binary: active vs inactive)
- **Activation score**: Can the description predict the *magnitude* of activation? (correlation)
- **Specificity score**: Does the description rule out tokens that DON'T activate the feature? (false positive rate)

A good description should have high detection AND high specificity.

In [ ]:
def score_description_simple(description_keywords, neuron_idx, all_acts, all_tokens):
    """Simple scoring: check if keyword presence correlates with activation.
    
    In practice, you'd use an LLM for scoring. This is a simplified version
    using keyword matching as a proxy.
    """
    predictions = []
    actuals = []
    
    for text_idx, (acts, tokens) in enumerate(zip(all_acts, all_tokens)):
        for pos in range(len(tokens)):
            # Simple prediction: does the context contain description keywords?
            context = "".join(tokens[max(0, pos-3):pos+3]).lower()
            predicted = any(kw.lower() in context for kw in description_keywords)
            actual_act = acts[pos, neuron_idx].item()
            
            predictions.append(1.0 if predicted else 0.0)
            actuals.append(actual_act)
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    
    # Threshold actuals to binary
    threshold = np.percentile(actuals, 90)  # Top 10% = "active"
    actuals_binary = (actuals > threshold).astype(float)
    
    # Compute metrics
    if predictions.sum() > 0 and actuals_binary.sum() > 0:
        # Precision: of predicted positives, how many are actually active?
        true_positives = (predictions * actuals_binary).sum()
        precision = true_positives / predictions.sum()
        
        # Recall: of actual positives, how many did we predict?
        recall = true_positives / actuals_binary.sum()
        
        # F1
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        # Correlation with continuous activations
        correlation = np.corrcoef(predictions, actuals)[0, 1]
    else:
        precision = recall = f1 = correlation = 0.0
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "correlation": correlation,
    }

# Example: score a hypothetical description
# (In practice, these keywords would come from an LLM's description)
example_descriptions = {
    neuron_stats[0][0]: ["science", "research", "study"],
    neuron_stats[1][0]: ["computer", "programming", "software"],
    neuron_stats[2][0]: ["world", "country", "global"],
}

print("Description Scoring (simplified keyword-based):\n")
for neuron_idx, keywords in example_descriptions.items():
    scores = score_description_simple(keywords, neuron_idx, all_acts, all_tokens)
    print(f"Neuron {neuron_idx} (keywords: {keywords}):")
    print(f"  Precision: {scores['precision']:.3f}")
    print(f"  Recall:    {scores['recall']:.3f}")
    print(f"  F1:        {scores['f1']:.3f}")
    print(f"  Correlation: {scores['correlation']:.3f}")
    print()

## 5. Scaling Auto-Interp — The Neuronpedia Approach

**Neuronpedia** (neuronpedia.org) has scaled automated interpretability to 50M+ features across multiple models. Their pipeline:

1. Train SAEs at every layer (or use pretrained like Gemma Scope)
2. For each feature, collect top activating examples from a large corpus
3. Run auto-interp using GPT-4/Claude to generate descriptions
4. Score descriptions automatically
5. Store everything in a searchable database with API access

**Key findings from scaled auto-interp:**
- ~70% of SAE features are interpretable (have descriptions that score well) — *this figure is approximate and varies by model, SAE width, and scoring methodology*
- Features organize into semantic clusters
- Many features are universal — appearing across different models
- Some features are "multi-scale": a broad feature splits into specific sub-features at wider SAE widths

**Limitations of auto-interp:**
- The interpreter LLM has its own biases (may over-interpret random activations)
- Short descriptions miss nuance (a feature might detect "words following 'the' in sentences about European geography" but get labeled just "European geography")
- Scoring with keywords or even LLM judgments is noisy
- We're using one black box (LLM) to interpret another (the model being studied)

## 6. Beyond Description — Automated Circuit Discovery

The frontier of automated interpretability goes beyond labeling individual features:

1. **Feature relationship discovery**: Automatically identify which features compose to form circuits. "Feature A (detects subject) + Feature B (detects verb tense) -> Feature C (subject-verb agreement)"

2. **Hypothesis generation**: LLMs generate hypotheses about model behavior, then automated tools test them. "I hypothesize that the model uses feature X for Y" -> run ablation -> confirm/reject.

3. **Cross-model comparison**: Run auto-interp on multiple models, align features by description similarity, identify universal vs model-specific features.

4. **Safety auditing at scale**: Automatically scan for features related to deception, bias, harmful content. Flag for human review.

The key tension: automated methods are scalable but less reliable. Human methods are reliable but don't scale. The field is working toward "human-in-the-loop" approaches where automation handles the bulk and humans verify critical findings.

## 7. Key Takeaways & Further Reading

**Key insights:**
- Auto-interp uses LLMs to describe and validate features found by SAEs
- The pipeline: collect examples -> describe -> score -> iterate
- ~70% of SAE features are interpretable (approximate; varies by methodology)
- Neuronpedia provides scaled auto-interp for community use
- Automated circuit discovery is the next frontier

**The meta-question**: When we use LLM_A to interpret LLM_B, what happens when LLM_A is wrong? Interpretability of the interpreter is an open problem.

**Further reading:**
- [Language models can explain neurons in language models](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) (Bills et al., 2023, OpenAI)
- [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) — includes auto-interp scoring methodology
- [Neuronpedia](https://www.neuronpedia.org/) — browse auto-generated feature descriptions
- [Automated Interpretability Agent](https://www.alignmentforum.org/posts/8ev6coxChSWcxCDy8) (Anthropic)